# 🛡️ GuardShield AI — High-VRAM Pre-Scan Training on Google Cloud TPU v5e-1
### Model Backbone: `microsoft/deberta-v3-base` (Guaranteed FP32 Stability)
**Hardware Target**: Google Cloud TPU v5e-1 (16GB HBM) / High-RAM Instance (47GB Host RAM)
**Dataset**: 15,304 Curated Samples (Stanford SPML, BeaverTails, Deepset, FEVER, XSTest v2)

---
### 📌 Execution Pipeline:
1. **Step 1**: Verify TPU Accelerator (`torch_xla` v5e-1 core)
2. **Step 2**: Mount Google Drive for Automatic Checkpoint Persistence
3. **Step 3**: Clone / Unpack Repository & Dataset Splits
4. **Step 4**: Execute Pure FP32 Training with Static Tensor Padding (No NaN Gradients)
5. **Step 5**: Run Camera-Ready Holdout Evaluations (In-Distribution, XSTest FPR, Long-Context)
6. **Step 6**: Package Model Weights for Deployment

## ⚡ Step 1: Verify Hardware Accelerator (TPU v5e-1 / GPU)

In [ ]:
import torch
print('PyTorch Version:', torch.__version__)

# Check for Google Cloud TPU
is_tpu = False
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    tpu_dev = xm.xla_device()
    print(f'✅ Active TPU Device: {tpu_dev} (Google Cloud TPU v5e-1)')
    is_tpu = True
except Exception as e:
    print(f'TPU not detected ({e}). Checking CUDA GPU...')

if not is_tpu:
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f'✅ Active CUDA GPU: {gpu_name} ({vram:.1f} GB VRAM)')
    else:
        print('⚠️ Running on Host CPU. (Ensure Runtime is set to TPU v5e or A100 GPU)')

## 💾 Step 2: Mount Google Drive for Automatic Checkpoint Backup

In [ ]:
import os
from pathlib import Path

# Optional: Connect Google Drive if you wish to persist weights to Drive
save_to_drive = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists('/content/drive/MyDrive'):
        drive_out_dir = Path('/content/drive/MyDrive/GuardShield_Models/deberta_v3_base_prescan_v4')
        drive_out_dir.mkdir(parents=True, exist_ok=True)
        save_to_drive = True
        print(f'✅ Google Drive connected! Model checkpoints will be saved to: {drive_out_dir}')
except Exception as e:
    print(f'⚠️ Google Drive mount skipped or cancelled ({e}).')

if not save_to_drive:
    local_out_dir = Path('models/deberta_v3_base_prescan_v4')
    local_out_dir.mkdir(parents=True, exist_ok=True)
    print(f'📁 Checkpoints will be saved to fast local Colab storage: {local_out_dir.resolve()}')

## 📂 Step 3: Unpack Repository & Dataset Splits

In [ ]:
import os, zipfile

# Unpack bundle if uploaded, or clone git repo
if os.path.exists('/content/GuardShield_AI_Lab_Bundle.zip'):
    print('Unpacking GuardShield_AI_Lab_Bundle.zip...')
    with zipfile.ZipFile('/content/GuardShield_AI_Lab_Bundle.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/GuardShield-AI')
elif not os.path.exists('/content/GuardShield-AI'):
    print('Cloning GuardShield-AI repository...')
    !git clone https://github.com/Shourya3113/GuardShield-AI.git /content/GuardShield-AI

%cd /content/GuardShield-AI
!pip install -q transformers sentencepiece accelerate datasets psutil

## 📊 Step 4: Verify the 15,304 Curated Benchmark Splits

In [ ]:
import json
from pathlib import Path

splits = [
    'data/massive_benchmark/stage1_train.json',
    'data/massive_benchmark/stage1_val.json',
    'data/massive_benchmark/stage1_test_indist.json',
    'data/massive_benchmark/stage1_test_xstest.json',
    'data/massive_benchmark/stage1_test_long_context.json'
]

print('Verifying Dataset Splits:')
for s in splits:
    p = Path(s)
    if p.exists():
        with open(p, 'r', encoding='utf-8') as f:
            cnt = len(json.load(f))
        size_mb = p.stat().st_size / (1024**2)
        print(f'  • {p.name:<32} -> {cnt:,} samples ({size_mb:.2f} MB)')
    else:
        print(f'  ❌ Missing: {s}')

## 🚀 Step 5: Execute High-VRAM Pre-Scan Training (FP32 Stable)
* **Architecture**: `microsoft/deberta-v3-base` (140M Total Parameters)
* **Context Window**: 512 Tokens with Static Tensor Padding
* **Batch Size**: 16 x 2 Gradient Accumulation (Effective Batch Size = 32)
* **Precision**: Full Precision FP32 (Completely avoids BF16 attention NaN gradient explosions)
* **Warmup**: 50 steps with AdamW weight decay 0.01

In [ ]:
import os

# Dynamically use Drive if mounted, otherwise local repository directory
out_dir = '/content/drive/MyDrive/GuardShield_Models/deberta_v3_base_prescan_v4' if os.path.exists('/content/drive/MyDrive') else 'models/deberta_v3_base_prescan_v4'
print(f'Target Output Directory: {out_dir}')

!python src/train_prescan_large.py \
    --model-name "microsoft/deberta-v3-base" \
    --max-length 512 \
    --batch-size 16 \
    --grad-accum 2 \
    --lr 2e-5 \
    --epochs 3 \
    --output-dir "$out_dir"

## 📈 Step 6: Inspect Camera-Ready Holdout Verification Metrics

In [ ]:
import os, json
from pathlib import Path

target_dir = Path('/content/drive/MyDrive/GuardShield_Models/deberta_v3_base_prescan_v4') if os.path.exists('/content/drive/MyDrive') else Path('models/deberta_v3_base_prescan_v4')
summary_file = target_dir / 'training_summary.json'

if summary_file.exists():
    with open(summary_file, 'r', encoding='utf-8') as f:
        res = json.load(f)
    print(json.dumps(res, indent=2))
else:
    print(f'Summary file not found at {summary_file}. Check local models/ directory.')

## 📦 Step 7: Package Final Weights for Deployment

In [ ]:
import os

if os.path.exists('/content/drive/MyDrive'):
    !tar -czvf /content/drive/MyDrive/GuardShield_Models/deberta_v3_base_prescan_v4.tar.gz -C /content/drive/MyDrive/GuardShield_Models/ deberta_v3_base_prescan_v4
    print('✅ Checkpoint archive saved to Google Drive!')
else:
    !tar -czvf deberta_v3_base_prescan_v4.tar.gz models/deberta_v3_base_prescan_v4/
    print('✅ Checkpoint archived to deberta_v3_base_prescan_v4.tar.gz in local storage. You can download it directly from the Colab Files panel!')